In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Train BEV v4 Swin GeomIM Cleaned

Self-contained notebook for a cleaned `v4` branch where we keep the same pipeline as the Swin GeomIM cleaned run, but replace only the image backbone with a `Swin Base` backbone initialized from `OccStudio/pretrain/swin-base-bevdet-stereo-geomim-512x1408.pth`.


In [ ]:
from src.utils.dist import barrier, cleanup_distributed, get_local_rank, get_rank, get_world_size, is_dist_enabled, is_main_process, setup_distributed


%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

cfg = {
    'run_dir': './runs/v4_swin_geomim_cleaned',
    'img_hw': (384, 704),
    'batch_size': 2,
    'val_batch_size': 1,
    'grad_accum': 16,
    'epochs': 20,
    'warmup_epochs': 3,
    'freeze_backbone_epochs': 2,
    'unfreeze_last_stages': 2,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'backbone_ckpt': './OccStudio/pretrain/swin-base-bevdet-stereo-geomim-512x1408.pth',
    'backbone_pretrain_img_size': (224, 224),
    'backbone_embed_dim': 128,
    'backbone_depths': (2, 2, 18, 2),
    'backbone_num_heads': (4, 8, 16, 32),
    'backbone_window_size': 16,
    'backbone_stage_index': 1,
    'backbone_stage_dims': (128, 256, 512, 1024),
    'backbone_proj_dim': 128,
    'use_ddp': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

setup_distributed()

random.seed(cfg['seed'] + get_rank())
np.random.seed(cfg['seed'] + get_rank())
torch.manual_seed(cfg['seed'] + get_rank())

if torch.cuda.is_available():
    device = torch.device(f'cuda:{get_local_rank()}') if is_dist_enabled() else torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if is_main_process():
    print('device:', device)
    if torch.cuda.is_available():
        print('cuda devices visible:', torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            try:
                print(i, torch.cuda.get_device_name(i))
            except RuntimeError as e:
                print(i, f'<cuda name unavailable: {e}>')
    print('world_size:', get_world_size(), '| rank:', get_rank(), '| local_rank:', get_local_rank())
    print('img_hw:', cfg['img_hw'])
    print('train batch_size(per gpu):', cfg['batch_size'], '| train workers:', cfg['num_workers'])
    print('val batch_size:', cfg['val_batch_size'], '| val workers:', cfg['val_num_workers'])
    print('backbone ckpt:', cfg['backbone_ckpt'])
    if torch.cuda.is_available() and torch.cuda.device_count() > 1 and not is_dist_enabled():
        print('NOTE: multiple GPUs are visible, but DDP is not active because torchrun env vars were not provided.')

if is_main_process():
    with open(RUN_DIR / 'config.json', 'w') as f:
        json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)


In [ ]:
from src.geometry import load_info_with_root, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [6]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))


{
  "merged_before_clean": 5000,
  "removed_empty_gt": 117,
  "after_empty_filter": 4883,
  "removed_by_dedup": 390,
  "clean_total": 4493,
  "dedup_groups": 173
}


,gt_occupancy_grid,header_ts,log_time,message_ts,/camera/inner/frontal/middle,/side/left/forward,/side/right/forward,/camera/inner/frontal/far,/camera/inner/frontal/middle/intrinsic_params,/camera/inner/frontal/middle/car_to_cam,/side/left/forward/intrinsic_params,/side/left/forward/car_to_cam,/side/right/forward/intrinsic_params,/side/right/forward/car_to_cam,/camera/inner/frontal/far/intrinsic_params,/camera/inner/frontal/far/car_to_cam,predicted_occupancy_grid,ride_date,ride_time,rover,scene_part_order,__data_root,__source_split,coverage,valid_count,pos_count
0,autonomy_yandex_dataset_train/static_grids/163...,1633857774533809000,12:18:58,1633857774533809000,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/predicted_static...,2021-10-10,12:18:58,orvy,0,autonomy_yandex_dataset_train,train,0.061972,1468,602
1,autonomy_yandex_dataset_train/static_grids/163...,1636812143899937000,15:34:56,1636812143899937000,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/predicted_static...,2021-11-13,15:34:56,shelly,0,autonomy_yandex_dataset_train,train,0.227077,5379,3752
2,autonomy_yandex_dataset_train/static_grids/163...,1633600207233930000,12:28:29,1633600207233930000,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/predicted_static...,2021-10-07,12:28:29,orvy,0,autonomy_yandex_dataset_train,train,0.225008,5330,1264


,kept_row_id,group_size,members
0,2271,2,"[2271, 3683]"
1,4101,2,"[4473, 4101]"
2,1366,3,"[306, 2710, 1366]"
3,98,5,"[1721, 2885, 209, 3630, 98]"
4,4151,2,"[4824, 4151]"
5,1703,3,"[526, 277, 1703]"
6,4867,3,"[4332, 4815, 4867]"
7,2549,2,"[3137, 2549]"
8,2112,2,"[2112, 3216]"
9,2330,2,"[2938, 2330]"


train_idx: 4273 val_idx: 220
num rover classes including Other: 26
top rover mapping: {'__other__': 0, 'orvy': 1, 'shelly': 2, 'lerita': 3, 'ward': 4, 'ravine': 5, 'greben': 6, 'lucky': 7, 'miro': 8, 'benzon': 9, 'natelio': 10, 'darron': 11, 'greton': 12, 'jurgie': 13, 'onipa': 14, 'targi': 15, 'kynde': 16, 'soan': 17, 'baland': 18, 'mika': 19, 'crozby': 20, 'litov': 21, 'hetera': 22, 'hankie': 23, 'stelard': 24, 'nikena': 25}


,rover,count,embedding_id,bucket
0,orvy,638,1,unique
1,shelly,496,2,unique
2,lerita,327,3,unique
3,ward,239,4,unique
4,ravine,208,5,unique
5,greben,194,6,unique
6,lucky,187,7,unique
7,miro,120,8,unique
8,benzon,114,9,unique
9,natelio,108,10,unique


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
from src.models.decoder import SmallUNet, _UNetBlock


class DropPath(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor

class PatchEmbed(nn.Module):
    def __init__(self, in_channels=3, embed_dims=128, kernel_size=4, stride=4, patch_norm=True):
        super().__init__()
        self.patch_size = (kernel_size, kernel_size)
        self.projection = nn.Conv2d(in_channels, embed_dims, kernel_size=kernel_size, stride=stride)
        self.norm = nn.LayerNorm(embed_dims) if patch_norm else None
        self.DH = None
        self.DW = None

    def forward(self, x):
        H, W = x.shape[2:]
        pad_h = (self.patch_size[0] - H % self.patch_size[0]) % self.patch_size[0]
        pad_w = (self.patch_size[1] - W % self.patch_size[1]) % self.patch_size[1]
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        x = self.projection(x)
        self.DH, self.DW = x.shape[2], x.shape[3]
        x = x.flatten(2).transpose(1, 2)
        if self.norm is not None:
            x = self.norm(x)
        return x

class PatchMerging(nn.Module):
    def __init__(self, in_channels, out_channels, stride=2, bias=False, patch_norm=True):
        super().__init__()
        self.out_channels = out_channels
        self.stride = stride
        self.sampler = nn.Unfold(kernel_size=stride, dilation=1, padding=0, stride=stride)
        sample_dim = stride ** 2 * in_channels
        self.norm = nn.LayerNorm(sample_dim) if patch_norm else None
        self.reduction = nn.Linear(sample_dim, out_channels, bias=bias)

    def forward(self, x, hw_shape):
        B, L, C = x.shape
        H, W = hw_shape
        assert L == H * W
        x = x.view(B, H, W, C).permute(0, 3, 1, 2)
        pad_h = H % self.stride
        pad_w = W % self.stride
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h))
        x = self.sampler(x).transpose(1, 2)
        if self.norm is not None:
            x = self.norm(x)
        x = self.reduction(x)
        down_hw_shape = ((H + self.stride - 1) // self.stride, (W + self.stride - 1) // self.stride)
        return x, down_hw_shape

class WindowMSA(nn.Module):
    def __init__(self, embed_dims, num_heads, window_size, qkv_bias=True, qk_scale=None, attn_drop_rate=0., proj_drop_rate=0.):
        super().__init__()
        self.embed_dims = embed_dims
        self.window_size = (window_size, window_size) if isinstance(window_size, int) else tuple(window_size)
        self.num_heads = num_heads
        head_dim = embed_dims // num_heads
        self.scale = qk_scale or head_dim ** -0.5
        size = (2 * self.window_size[0] - 1) * (2 * self.window_size[1] - 1)
        self.relative_position_bias_table = nn.Parameter(torch.zeros(size, num_heads))
        Wh, Ww = self.window_size
        coords_h = torch.arange(Wh)
        coords_w = torch.arange(Ww)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing='ij'))
        coords_flatten = coords.flatten(1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += Wh - 1
        relative_coords[:, :, 1] += Ww - 1
        relative_coords[:, :, 0] *= 2 * Ww - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer('relative_position_index', relative_position_index)
        self.qkv = nn.Linear(embed_dims, embed_dims * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop_rate)
        self.proj = nn.Linear(embed_dims, embed_dims)
        self.proj_drop = nn.Dropout(proj_drop_rate)
        self.softmax = nn.Softmax(dim=-1)
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def forward(self, x, mask=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.reshape(-1)]
        relative_position_bias = relative_position_bias.view(N, N, -1).permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class ShiftWindowMSA(nn.Module):
    def __init__(self, embed_dims, num_heads, window_size, shift_size=0, qkv_bias=True, qk_scale=None, attn_drop_rate=0., proj_drop_rate=0., drop_path_rate=0.):
        super().__init__()
        self.window_size = window_size
        self.shift_size = shift_size
        self.w_msa = WindowMSA(embed_dims, num_heads, window_size, qkv_bias, qk_scale, attn_drop_rate, proj_drop_rate)
        self.drop = DropPath(drop_path_rate)

    def window_partition(self, x):
        B, H, W, C = x.shape
        ws = self.window_size
        x = x.view(B, H // ws, ws, W // ws, ws, C)
        windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, ws, ws, C)
        return windows

    def window_reverse(self, windows, H, W):
        ws = self.window_size
        B = int(windows.shape[0] / (H * W / ws / ws))
        x = windows.view(B, H // ws, W // ws, ws, ws, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
        return x

    def forward(self, query, hw_shape):
        B, L, C = query.shape
        H, W = hw_shape
        query = query.view(B, H, W, C)
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r or pad_b:
            query = F.pad(query, (0, 0, 0, pad_r, 0, pad_b))
        H_pad, W_pad = query.shape[1], query.shape[2]
        if self.shift_size > 0:
            shifted_query = torch.roll(query, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            img_mask = torch.zeros((1, H_pad, W_pad, 1), device=query.device)
            h_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size), slice(-self.window_size, -self.shift_size), slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1
            mask_windows = self.window_partition(img_mask).view(-1, self.window_size * self.window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        else:
            shifted_query = query
            attn_mask = None
        query_windows = self.window_partition(shifted_query).view(-1, self.window_size * self.window_size, C)
        attn_windows = self.w_msa(query_windows, mask=attn_mask)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = self.window_reverse(attn_windows, H_pad, W_pad)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
        if pad_r or pad_b:
            x = x[:, :H, :W, :].contiguous()
        x = x.view(B, H * W, C)
        return self.drop(x)

class MLP(nn.Module):
    def __init__(self, embed_dims, feedforward_channels, drop_rate=0.0, drop_path_rate=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(embed_dims, feedforward_channels), nn.GELU(), nn.Dropout(drop_rate)),
            nn.Linear(feedforward_channels, embed_dims),
        ])
        self.drop = DropPath(drop_path_rate)
        self.out_drop = nn.Dropout(drop_rate)

    def forward(self, x, identity=None):
        out = self.layers[0](x)
        out = self.layers[1](out)
        out = self.out_drop(out)
        out = self.drop(out)
        return (identity if identity is not None else x) + out

class SwinBlock(nn.Module):
    def __init__(self, embed_dims, num_heads, feedforward_channels, window_size=16, shift=False, qkv_bias=True, qk_scale=None, drop_rate=0., attn_drop_rate=0., drop_path_rate=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dims)
        self.attn = ShiftWindowMSA(
            embed_dims=embed_dims,
            num_heads=num_heads,
            window_size=window_size,
            shift_size=window_size // 2 if shift else 0,
            qkv_bias=qkv_bias,
            qk_scale=qk_scale,
            attn_drop_rate=attn_drop_rate,
            proj_drop_rate=drop_rate,
            drop_path_rate=drop_path_rate,
        )
        self.norm2 = nn.LayerNorm(embed_dims)
        self.ffn = MLP(embed_dims, feedforward_channels, drop_rate=drop_rate, drop_path_rate=drop_path_rate)

    def forward(self, x, hw_shape):
        identity = x
        x = self.norm1(x)
        x = self.attn(x, hw_shape)
        x = x + identity
        identity = x
        x = self.norm2(x)
        x = self.ffn(x, identity=identity)
        return x

class SwinBlockSequence(nn.Module):
    def __init__(self, embed_dims, num_heads, feedforward_channels, depth, window_size=16, qkv_bias=True, qk_scale=None, drop_rate=0., attn_drop_rate=0., drop_path_rate=0., downsample=None, with_cp=True):
        super().__init__()
        if not isinstance(drop_path_rate, list):
            drop_path_rate = [float(drop_path_rate) for _ in range(depth)]
        self.blocks = nn.ModuleList([
            SwinBlock(
                embed_dims=embed_dims,
                num_heads=num_heads,
                feedforward_channels=feedforward_channels,
                window_size=window_size,
                shift=(i % 2 == 1),
                qkv_bias=qkv_bias,
                qk_scale=qk_scale,
                drop_rate=drop_rate,
                attn_drop_rate=attn_drop_rate,
                drop_path_rate=drop_path_rate[i],
            )
            for i in range(depth)
        ])
        self.downsample = downsample
        self.with_cp = with_cp

    def forward(self, x, hw_shape):
        for block in self.blocks:
            if self.with_cp and self.training:
                x = torch.utils.checkpoint.checkpoint(block, x, hw_shape, use_reentrant=False)
            else:
                x = block(x, hw_shape)
        if self.downsample is not None:
            x_down, down_hw_shape = self.downsample(x, hw_shape)
            return x_down, down_hw_shape, x, hw_shape
        return x, hw_shape, x, hw_shape

class SwinTransformerMMCVLike(nn.Module):
    def __init__(self, pretrain_img_size=(512, 1408), in_channels=3, embed_dims=128, patch_size=4, window_size=16, mlp_ratio=4, depths=(2, 2, 18, 2), num_heads=(4, 8, 16, 32), strides=(4, 2, 2, 2), out_indices=(1,), qkv_bias=True, qk_scale=None, patch_norm=True, drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1, use_abs_pos_embed=True, with_cp=True):
        super().__init__()
        self.pretrain_img_size = pretrain_img_size
        self.embed_dims = embed_dims
        self.patch_size = patch_size
        self.out_indices = out_indices
        self.use_abs_pos_embed = use_abs_pos_embed
        self.patch_embed = PatchEmbed(in_channels=in_channels, embed_dims=embed_dims, kernel_size=patch_size, stride=strides[0], patch_norm=patch_norm)
        if self.use_abs_pos_embed:
            ph = pretrain_img_size[0] // patch_size
            pw = pretrain_img_size[1] // patch_size
            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, ph * pw, embed_dims))
            nn.init.trunc_normal_(self.absolute_pos_embed, std=0.02)
        self.drop_after_pos = nn.Dropout(drop_rate)
        total_depth = sum(depths)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, total_depth)]
        self.stages = nn.ModuleList()
        in_dim = embed_dims
        for i in range(len(depths)):
            downsample = None
            if i < len(depths) - 1:
                downsample = PatchMerging(in_dim, 2 * in_dim, stride=strides[i + 1], patch_norm=patch_norm)
            stage = SwinBlockSequence(
                embed_dims=in_dim,
                num_heads=num_heads[i],
                feedforward_channels=mlp_ratio * in_dim,
                depth=depths[i],
                window_size=window_size,
                qkv_bias=qkv_bias,
                qk_scale=qk_scale,
                drop_rate=drop_rate,
                attn_drop_rate=attn_drop_rate,
                drop_path_rate=dpr[:depths[i]],
                downsample=downsample,
                with_cp=with_cp,
            )
            self.stages.append(stage)
            dpr = dpr[depths[i]:]
            if downsample is not None:
                in_dim = downsample.out_channels
        self.num_features = [embed_dims * (2 ** i) for i in range(len(depths))]
        for i in out_indices:
            self.add_module(f'norm{i}', nn.LayerNorm(self.num_features[i]))

    def _get_abs_pos_embed(self, hw_shape):
        abs_pos = self.absolute_pos_embed
        Hp0 = self.pretrain_img_size[0] // self.patch_size
        Wp0 = self.pretrain_img_size[1] // self.patch_size
        Hp, Wp = hw_shape
        if Hp == Hp0 and Wp == Wp0:
            return abs_pos
        abs_pos = abs_pos.view(1, Hp0, Wp0, -1).permute(0, 3, 1, 2).contiguous()
        abs_pos = F.interpolate(abs_pos, size=(Hp, Wp), mode='bicubic', align_corners=False)
        abs_pos = abs_pos.permute(0, 2, 3, 1).reshape(1, Hp * Wp, -1).contiguous()
        return abs_pos

    def forward(self, x):
        x = self.patch_embed(x)
        hw_shape = (self.patch_embed.DH, self.patch_embed.DW)
        if self.use_abs_pos_embed:
            x = x + self._get_abs_pos_embed(hw_shape).to(dtype=x.dtype)
        x = self.drop_after_pos(x)
        outs = []
        for i, stage in enumerate(self.stages):
            x, hw_shape, out, out_hw_shape = stage(x, hw_shape)
            if i in self.out_indices:
                norm_layer = getattr(self, f'norm{i}')
                out = norm_layer(out)
                out = out.view(-1, out_hw_shape[0], out_hw_shape[1], self.num_features[i]).permute(0, 3, 1, 2).contiguous()
                outs.append(out)
        return outs

class _SwinGeomIMBackbone(nn.Module):
    def __init__(self, ckpt_path, pretrain_img_size=(512, 1408), embed_dims=128, depths=(2, 2, 18, 2), num_heads=(4, 8, 16, 32), window_size=16, stage_index=1, proj_dim=128, with_cp=True):
        super().__init__()
        self.ckpt_path = Path(ckpt_path)
        self.stage_index = stage_index
        self.swin = SwinTransformerMMCVLike(
            pretrain_img_size=pretrain_img_size,
            embed_dims=embed_dims,
            patch_size=4,
            window_size=window_size,
            mlp_ratio=4,
            depths=depths,
            num_heads=num_heads,
            strides=(4, 2, 2, 2),
            out_indices=(stage_index,),
            qkv_bias=True,
            qk_scale=None,
            patch_norm=True,
            drop_rate=0.0,
            attn_drop_rate=0.0,
            drop_path_rate=0.1,
            use_abs_pos_embed=True,
            with_cp=with_cp,
        )
        stage_dims = [embed_dims * (2 ** i) for i in range(len(depths))]
        self.proj = nn.Conv2d(stage_dims[stage_index], proj_dim, 1)
        self._load_pretrained()

    def _load_pretrained(self):
        if not self.ckpt_path.exists():
            raise FileNotFoundError(f'Swin checkpoint not found: {self.ckpt_path}')
        ckpt = torch.load(self.ckpt_path, map_location='cpu')
        state = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
        raw_backbone_state = {}
        for k, v in state.items():
            if k.startswith('img_backbone.'):
                key = k[len('img_backbone.'):]
                if key.endswith('relative_position_index'):
                    continue
                raw_backbone_state[key] = v

        model_state = self.swin.state_dict()
        backbone_state = {}
        skipped = []

        for k, v in raw_backbone_state.items():
            if k not in model_state:
                skipped.append((k, 'missing_in_model'))
                continue

            target = model_state[k]

            if k == 'absolute_pos_embed':
                if v.ndim == 4:
                    v = F.interpolate(v.float(), size=(56, 56), mode='bicubic', align_corners=False)
                    v = v.flatten(2).transpose(1, 2).contiguous()
                if v.shape != target.shape:
                    src_tokens = v.shape[1]
                    src_hw = int(round(src_tokens ** 0.5))
                    if src_hw * src_hw != src_tokens:
                        skipped.append((k, f'unexpected_abs_pos_shape_{tuple(v.shape)}'))
                        continue
                    v = v.view(1, src_hw, src_hw, v.shape[-1]).permute(0, 3, 1, 2).contiguous()
                    dst_hw = int(round(target.shape[1] ** 0.5))
                    if dst_hw * dst_hw != target.shape[1]:
                        skipped.append((k, f'unexpected_target_abs_pos_shape_{tuple(target.shape)}'))
                        continue
                    v = F.interpolate(v.float(), size=(dst_hw, dst_hw), mode='bicubic', align_corners=False)
                    v = v.permute(0, 2, 3, 1).reshape_as(target).contiguous()
                backbone_state[k] = v.to(dtype=target.dtype)
                continue

            if 'relative_position_bias_table' in k and v.shape != target.shape:
                src_len, num_heads = v.shape
                dst_len, dst_heads = target.shape
                if num_heads != dst_heads:
                    skipped.append((k, f'num_heads_mismatch_{tuple(v.shape)}_{tuple(target.shape)}'))
                    continue
                src_hw = int(round(src_len ** 0.5))
                dst_hw = int(round(dst_len ** 0.5))
                v = v.permute(1, 0).reshape(1, num_heads, src_hw, src_hw)
                v = F.interpolate(v.float(), size=(dst_hw, dst_hw), mode='bicubic', align_corners=False)
                v = v.reshape(num_heads, dst_len).permute(1, 0).contiguous()
                backbone_state[k] = v.to(dtype=target.dtype)
                continue

            if v.shape != target.shape:
                skipped.append((k, f'shape_mismatch_{tuple(v.shape)}_{tuple(target.shape)}'))
                continue

            backbone_state[k] = v.to(dtype=target.dtype)

        missing, unexpected = self.swin.load_state_dict(backbone_state, strict=False)
        print('loaded Swin backbone from', self.ckpt_path)
        print('  raw backbone keys:', len(raw_backbone_state))
        print('  loaded keys:', len(backbone_state))
        print('  skipped keys:', len(skipped))
        print('  missing keys:', len(missing))
        print('  unexpected keys:', len(unexpected))
        if len(skipped):
            print('  sample skipped:', skipped[:10])
        if len(missing):
            print('  sample missing:', missing[:10])
        if len(unexpected):
            print('  sample unexpected:', unexpected[:10])
        del ckpt
        del state
        del raw_backbone_state
        del model_state
        gc.collect()

    def forward(self, x):
        outs = self.swin(x)
        feat = outs[0]
        feat = self.proj(feat)
        return feat

class MultiCamBEVv4SwinGeomIMClean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 ckpt_path: str | Path = './OccStudio/pretrain/swin-base-bevdet-stereo-geomim-512x1408.pth',
                 pretrain_img_size=(512, 1408),
                 embed_dims=128,
                 depths=(2, 2, 18, 2),
                 num_heads=(4, 8, 16, 32),
                 window_size=16,
                 stage_index=1,
                 proj_dim=128):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _SwinGeomIMBackbone(
            ckpt_path=ckpt_path,
            pretrain_img_size=pretrain_img_size,
            embed_dims=embed_dims,
            depths=depths,
            num_heads=num_heads,
            window_size=window_size,
            stage_index=stage_index,
            proj_dim=proj_dim,
            with_cp=True,
        )
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(proj_dim, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feat = self.backbone(images.reshape(B * N, C_, Hi, Wi)).float()
        feat = torch.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)

        feat = self.feat_proj(feat.float())
        feat = torch.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)

        vox = self.ego_voxels.to(images.device, dtype=images.dtype)
        vox = vox.unsqueeze(0).unsqueeze(0).repeat(B, N, 1, 1, 1, 1)

        cam_xyz1 = torch.einsum('bnij,bnzyxj->bnzyxi', car2cams.float(), vox)
        cam_xyz1 = torch.nan_to_num(cam_xyz1, nan=0.0, posinf=0.0, neginf=0.0)

        Xc = cam_xyz1[..., 0]
        Yc = cam_xyz1[..., 1]
        Zc = cam_xyz1[..., 2].clamp(min=1e-4)

        fx = intrinsics[..., 0, 0].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        fy = intrinsics[..., 1, 1].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        cx = intrinsics[..., 0, 2].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
        cy = intrinsics[..., 1, 2].unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)

        u = fx * (Xc / Zc) + cx
        v = fy * (Yc / Zc) + cy

        gx = (u / max(Wi - 1, 1)) * 2 - 1
        gy = (v / max(Hi - 1, 1)) * 2 - 1
        grid = torch.stack([gx, gy], dim=-1)
        grid = grid.permute(0, 1, 3, 4, 2, 5).contiguous()
        grid = grid.reshape(B * N, self.bev_h * self.bev_w * len(self.z_levels), 1, 2)
        grid = torch.nan_to_num(grid, nan=0.0, posinf=2.0, neginf=-2.0)

        sampled = F.grid_sample(
            feat.float(),
            grid.float(),
            mode='bilinear',
            padding_mode='zeros',
            align_corners=True,
        )
        sampled = torch.nan_to_num(sampled, nan=0.0, posinf=0.0, neginf=0.0)

        sampled = sampled.view(B, N, 64, len(self.z_levels), self.bev_h, self.bev_w)
        bev = sampled.mean(dim=1)
        bev = bev.reshape(B, 64 * len(self.z_levels), self.bev_h, self.bev_w)
        bev = torch.nan_to_num(bev, nan=0.0, posinf=0.0, neginf=0.0)

        rover_vec = self.rover_mlp(self.rover_embed(rover_ids)).unsqueeze(-1).unsqueeze(-1)
        rover_map = rover_vec.expand(-1, -1, self.bev_h, self.bev_w)
        bev = torch.cat([bev, rover_map], dim=1)

        logits = self.bev_decoder(bev)
        logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
        return logits


In [ ]:
from src.eval import evaluate_iou
from src.losses import CompoundLossV2, _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch, streaming_threshold_sweep
from src.utils.training import cleanup_cuda, unwrap_model



In [19]:
ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

train_sampler_warm = None
train_sampler_aug = None
if is_dist_enabled():
    train_sampler_warm = DistributedSampler(ds_train_warm, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)
    train_sampler_aug = DistributedSampler(ds_train_aug, num_replicas=get_world_size(), rank=get_rank(), shuffle=True, drop_last=True)

loader_warm = DataLoader(
    ds_train_warm,
    batch_size=cfg['batch_size'],
    shuffle=(train_sampler_warm is None),
    sampler=train_sampler_warm,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_train = DataLoader(
    ds_train_aug,
    batch_size=cfg['batch_size'],
    shuffle=(train_sampler_aug is None),
    sampler=train_sampler_aug,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_val = None
if is_main_process():
    loader_val = DataLoader(
        ds_val,
        batch_size=cfg['val_batch_size'],
        shuffle=False,
        num_workers=cfg['val_num_workers'],
        pin_memory=(device.type == 'cuda'),
    )

if is_main_process():
    print('loader_warm batch_size:', loader_warm.batch_size, '| workers:', loader_warm.num_workers)
    print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)
    if loader_val is not None:
        print('loader_val batch_size:', loader_val.batch_size, '| workers:', loader_val.num_workers)

    sample = ds_train_warm[0]
    for k, v in sample.items():
        if isinstance(v, torch.Tensor):
            print(k, tuple(v.shape), v.dtype)
        else:
            print(k, type(v), v)



loader_warm batch_size: 2 | workers: 4
loader_train batch_size: 2 | workers: 4
loader_val batch_size: 1 | workers: 0
images (4, 3, 384, 704) torch.float32
intrinsics (4, 3, 3) torch.float32
car2cams (4, 4, 4) torch.float32
rover_id () torch.int64
info_idx <class 'int'> 0
gt (1, 188, 126) torch.int64


In [ ]:
from src.utils.training import freeze_all_backbone, update_ema


base_model = MultiCamBEVv4SwinGeomIMClean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    freeze_backbone=False,
    ckpt_path=cfg['backbone_ckpt'],
    pretrain_img_size=cfg['backbone_pretrain_img_size'],
    embed_dims=cfg['backbone_embed_dim'],
    depths=cfg['backbone_depths'],
    num_heads=cfg['backbone_num_heads'],
    window_size=cfg['backbone_window_size'],
    stage_index=cfg['backbone_stage_index'],
    proj_dim=cfg['backbone_proj_dim'],
).to(device)

def unfreeze_last_stages(model, n_last_stages=2):
    core = unwrap_model(model)
    freeze_all_backbone(core)
    stages = list(core.backbone.swin.stages)
    for stage in stages[-n_last_stages:]:
        for p in stage.parameters():
            p.requires_grad = True
    if hasattr(core.backbone.swin, 'absolute_pos_embed'):
        core.backbone.swin.absolute_pos_embed.requires_grad = True
    for p in core.backbone.proj.parameters():
        p.requires_grad = True

freeze_all_backbone(base_model)

if is_dist_enabled():
    model = DDP(
        base_model,
        device_ids=[get_local_rank()],
        output_device=get_local_rank(),
        broadcast_buffers=False,
        find_unused_parameters=False,
    )
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params = []
embed_params = []
other_params = []
for name, p in core_model.named_parameters():
    if name.startswith('backbone.swin.') or name.startswith('backbone.proj.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

if is_main_process():
    print('params M:', sum(p.numel() for p in core_model.parameters()) / 1e6)
    print('backbone frozen at start:', not any(p.requires_grad for p in core_model.backbone.swin.parameters()))

    with torch.no_grad():
        batch = next(iter(loader_warm))
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)

        feat = core_model.backbone(images.reshape(-1, images.shape[2], images.shape[3], images.shape[4]))
        out = core_model(images, intr, c2c, rover_id)

        expected_h = cfg['img_hw'][0] // 8
        expected_w = cfg['img_hw'][1] // 8
        print('expected stride-8 grid:', (expected_h, expected_w))
        print('backbone feature shape:', tuple(feat.shape))
        print('sanity output shape:', tuple(out.shape))

cleanup_cuda()
barrier()


In [21]:
core = unwrap_model(model)
batch = next(iter(loader_warm))

images = batch['images'].to(device)
intr = batch['intrinsics'].to(device)
c2c = batch['car2cams'].to(device)
rover_id = batch['rover_id'].to(device)

x = images.reshape(-1, images.shape[2], images.shape[3], images.shape[4])

with torch.inference_mode():
    sw = core.backbone.swin

    x = sw.patch_embed(x)
    hw = (sw.patch_embed.DH, sw.patch_embed.DW)
    print('after patch_embed:', torch.isfinite(x).all().item(), x.min().item(), x.max().item(), hw)

    if sw.use_abs_pos_embed:
        x = x + sw._get_abs_pos_embed(hw).to(dtype=x.dtype)
    print('after abs pos:', torch.isfinite(x).all().item(), x.min().item(), x.max().item())

    x = sw.drop_after_pos(x)
    print('after drop_after_pos:', torch.isfinite(x).all().item(), x.min().item(), x.max().item())

    for i, stage in enumerate(sw.stages):
        x, hw, out, out_hw = stage(x, hw)
        print(f'after stage{i}:', torch.isfinite(x).all().item(), x.min().item(), x.max().item(), 'hw=', hw)

    feat = out
    print('stage output before norm/proj:', torch.isfinite(feat).all().item(), feat.min().item(), feat.max().item())

    feat = core.backbone.proj(feat)
    print('after backbone.proj:', torch.isfinite(feat).all().item(), feat.min().item(), feat.max().item())

    feat = core.feat_proj(feat)
    print('after feat_proj:', torch.isfinite(feat).all().item(), feat.min().item(), feat.max().item())

    logits = core(images, intr, c2c, rover_id)
    print('final logits:', torch.isfinite(logits).all().item(), logits.min().item(), logits.max().item())


after patch_embed: True -9.131632804870605 8.539260864257812 (96, 176)
after abs pos: True -8.469563484191895 9.342641830444336
after drop_after_pos: True -8.469563484191895 9.342641830444336
after stage0: True -11.508835792541504 11.646700859069824 hw= (48, 88)
after stage1: True -20.817020416259766 20.652650833129883 hw= (24, 44)
after stage2: True -19.205827713012695 15.702770233154297 hw= (12, 22)
after stage3: True -48.064361572265625 51.356571197509766 hw= (12, 22)
stage output before norm/proj: True -48.064361572265625 51.356571197509766


RuntimeError: Given groups=1, weight of size [128, 256, 1, 1], expected input[1, 8, 264, 1024] to have 256 channels, but got 8 channels instead

In [22]:
log = []
best_iou = -1.0
best_ema_iou = -1.0
start = time.time()
backbone_unfrozen = False

for epoch in range(cfg['epochs']):
    if train_sampler_warm is not None:
        train_sampler_warm.set_epoch(epoch)
    if train_sampler_aug is not None:
        train_sampler_aug.set_epoch(epoch)

    if (not backbone_unfrozen) and epoch >= cfg['freeze_backbone_epochs']:
        unfreeze_last_stages(model, n_last_stages=cfg['unfreeze_last_stages'])
        backbone_unfrozen = True
        if is_main_process():
            print(f'unfroze last {cfg["unfreeze_last_stages"]} Swin stages at epoch {epoch:02d}')

    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]', disable=not is_main_process())
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id)
        logits = logits.float()
        loss, parts = loss_fn(logits, gt)
        loss = loss / cfg['grad_accum']

        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite loss detected at epoch={epoch} step={step}: {loss.item()}')

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if is_main_process() and step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(core_model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()
    barrier()

    val_iou_05 = None
    val_iou_05_ema = None
    if is_main_process():
        cleanup_cuda()
        print('start val model @0.5')
        val_iou_05 = evaluate_iou(core_model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

        cleanup_cuda()
        print('start val ema @0.5')
        val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
        cleanup_cuda()

        elapsed = (time.time() - start) / 60
        backbone_grad_enabled = any(p.requires_grad for p in core_model.backbone.swin.parameters())

        row = {
            'epoch': epoch,
            'loss': float(np.mean(losses)) if len(losses) else None,
            'val_iou_05': float(val_iou_05),
            'val_iou_05_ema': float(val_iou_05_ema),
            'backbone_trainable': bool(backbone_grad_enabled),
            'minutes': float(elapsed),
        }

        print(
            f"ep{epoch:02d} | loss {np.mean(losses):.3f} | "
            f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
            f"backbone_trainable={backbone_grad_enabled} | {elapsed:.1f}m"
        )

        if val_iou_05 > best_iou:
            best_iou = val_iou_05
            torch.save({
                'model_type': 'v4_swin_geomim_cleaned',
                'model': core_model.state_dict(),
                'epoch': epoch,
                'best_iou': float(val_iou_05),
                'best_t': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new best model:', val_iou_05)

        if val_iou_05_ema > best_ema_iou:
            best_ema_iou = val_iou_05_ema
            torch.save({
                'model_type': 'v4_swin_geomim_cleaned',
                'model': core_model.state_dict(),
                'ema': ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': float(val_iou_05_ema),
                'best_t_ema': 0.5,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new best ema:', val_iou_05_ema)

        log.append(row)
        pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
        torch.save({
            'model_type': 'v4_swin_geomim_cleaned',
            'model': core_model.state_dict(),
            'ema': ema_model.state_dict(),
            'epoch': epoch,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'last.pt')

    barrier()



ep06 [AUG]: 100%|██████████| 2136/2136 [24:59<00:00,  1.42it/s, bce=1.41, loss=0.981]


start val model @0.5


start val ema @0.5


ep06 | loss 0.986 | val@0.5 0.4566 | ema@0.5 0.4534 | backbone_trainable=True | 179.7m
  new best model: 0.4565861013889806
  new best ema: 0.45342061879050616


ep07 [AUG]:   2%|▏         | 50/2136 [00:36<25:04,  1.39it/s, bce=1.95, loss=0.988] 


KeyboardInterrupt: 

In [ ]:
if is_main_process():
    ckpt = torch.load(RUN_DIR / 'ema_best.pt', map_location=device) if (RUN_DIR / 'ema_best.pt').exists() else torch.load(RUN_DIR / 'best.pt', map_location=device)
    model_eval = MultiCamBEVv4SwinGeomIMClean(
        num_rover_classes=len(rover_vocab),
        rover_emb_dim=cfg['rover_emb_dim'],
        rover_cond_dim=cfg['rover_cond_dim'],
        freeze_backbone=False,
        ckpt_path=cfg['backbone_ckpt'],
        pretrain_img_size=cfg['backbone_pretrain_img_size'],
        embed_dims=cfg['backbone_embed_dim'],
        depths=cfg['backbone_depths'],
        num_heads=cfg['backbone_num_heads'],
        window_size=cfg['backbone_window_size'],
        stage_index=cfg['backbone_stage_index'],
        proj_dim=cfg['backbone_proj_dim'],
    ).to(device)
    state = ckpt['ema'] if 'ema' in ckpt else ckpt['model']
    model_eval.load_state_dict(state, strict=False)

    thresholds = [round(x, 2) for x in np.arange(0.20, 0.81, 0.02)]
    iou_by_t = streaming_threshold_sweep(model_eval, loader_val, thresholds)
    best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
    print('best threshold:', best_t, 'best_iou:', best_iou)
    for t, iou in iou_by_t.items():
        marker = ' <- best' if t == best_t else ''
        print(f't={t:.2f}  iou={iou:.4f}{marker}')
else:
    print('post-train threshold sweep is skipped on non-main DDP ranks')



## Notes

- `clean_merged_info(...)` only writes caches and reports into `RUN_DIR / preproc_cache`.
- The original dataset folders are never modified.
- The `rover_vocab` is built strictly from the cleaned training split, so rare rovers and unseen rovers map to `Other=0`.
- Validation is selected from the merged cleaned pool and targets about 200 samples while staying group-aware by `(rover, ride_date)`.
- Swin GeomIM weights are expected to come from `OccStudio/pretrain/swin-base-bevdet-stereo-geomim-512x1408.pth`, and the notebook will fail early on the sanity-check cell if the checkpoint is missing.
- DDP is activated only when the notebook runs under `torchrun` with `WORLD_SIZE > 1`; otherwise it falls back to single-process training.
- Validation and checkpoint saving run only on `rank 0` in DDP mode.
- To make Swin fine-tuning fit on `2 x T4`, the notebook uses per-GPU batch size `2`, `grad_accum = 16`, and unfreezes only the last Swin stages instead of the full backbone.

